# Day 24 - SCD Type 1 and Type 2

**Topic:** SCD Type 1 and Type 2  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจและ implement basic dimension change handling ด้วย SCD Type 1 overwrite และ SCD Type 2 history tracking ใน PySpark/Lakehouse table

วันนี้เราจะฝึกจัดการ dimension data ที่เปลี่ยนแปลงตามเวลา เช่น customer profile, status, city, loyalty tier โดยเปรียบเทียบว่าเมื่อไรควรใช้ **SCD Type 1** เพื่อเก็บเฉพาะค่าล่าสุด และเมื่อไรควรใช้ **SCD Type 2** เพื่อเก็บประวัติการเปลี่ยนแปลงย้อนหลังสำหรับ reporting และ audit

## Goal of the day

1. แยกให้ออกว่า **SCD Type 1** และ **SCD Type 2** ต่างกันอย่างไร
2. อธิบายได้ว่า dimension table ต้องมี **natural key**, **surrogate key**, `valid_from`, `valid_to` และ `is_current` เพื่ออะไร
3. ใช้ PySpark เพื่อเตรียม source update batch และ deduplicate ให้เหลือ latest record ต่อ business key
4. Implement Type 1 overwrite ด้วย `MERGE INTO` บน Lakehouse table
5. Implement Type 2 แบบ basic flow: detect change → expire current record → insert new current version

## Why it matters in production

ใน production pipeline ข้อมูล dimension เช่น customer, product, branch, policy หรือ account มักเปลี่ยนเรื่อย ๆ แต่คำถามทาง business ไม่ได้มีแค่ “ค่าปัจจุบันคืออะไร” เสมอไป

- ถ้าต้องการแค่ค่าล่าสุด เช่น เบอร์โทรล่าสุด, email ล่าสุด อาจใช้ **SCD Type 1** ได้
- ถ้าต้องการตอบคำถามย้อนหลัง เช่น “ตอนวันที่ขาย policy ลูกค้าอยู่ tier ไหน” ต้องใช้ **SCD Type 2**
- ถ้า update dimension ผิด pattern อาจทำให้รายงานย้อนหลังผิดโดยไม่รู้ตัว
- Type 2 ต้องคุมให้มี current record แค่ 1 record ต่อ natural key
- Source update batch ต้อง deduplicate ก่อน merge ไม่อย่างนั้นอาจเกิดหลาย version โดยไม่ตั้งใจ

Production takeaway:

> SCD ไม่ใช่แค่การ update table แต่คือการตัดสินใจว่า business ต้องการ “current state” หรือ “historical truth”

## Key concepts

| Concept | Meaning |
|---|---|
| SCD | Slowly Changing Dimension คือ pattern สำหรับจัดการ dimension records ที่เปลี่ยนตามเวลา |
| SCD Type 1 | Update/overwrite record เดิม เก็บเฉพาะค่าล่าสุด ไม่เก็บประวัติ |
| SCD Type 2 | ปิด record เก่า แล้ว insert record ใหม่ เพื่อเก็บ history |
| Natural key | Business key จาก source เช่น `customer_id` |
| Surrogate key | Key ที่ใช้แทนแต่ละ version ของ dimension record เช่น `customer_sk` |
| Current record | Record ล่าสุดของ key นั้น มัก mark ด้วย `is_current = true` |
| History record | Record เก่าที่ถูกปิดช่วงเวลาแล้ว และมี `is_current = false` |
| Effective date | วันที่ record version นั้นเริ่มมีผล เช่น `valid_from` |
| End date | วันที่ record version นั้นหมดอายุ เช่น `valid_to` |
| Change detection | วิธีเช็คว่า attributes เปลี่ยนจริงไหม อาจ compare column ตรง ๆ หรือใช้ attribute hash |

## Concept explanation

### SCD Type 1 คืออะไร?

**SCD Type 1** คือการ update record เดิมให้เป็นค่าล่าสุด โดยไม่เก็บค่าก่อนหน้า

เหมาะกับ field ที่ business ไม่ต้องการ history เช่น:

- corrected spelling ของชื่อ
- email ล่าสุด
- phone ล่าสุด
- status ล่าสุดในบาง use case

ตัวอย่าง:

```text
Before: C001 | Alice | Bangkok
After : C001 | Alice | Chiang Mai
```

หลัง update แล้วเราจะไม่รู้จาก table นี้ว่า C001 เคยอยู่ Bangkok มาก่อน

> Type 1 = simple current-state table

### SCD Type 2 คืออะไร?

**SCD Type 2** คือการเก็บประวัติ dimension โดยไม่ overwrite record เดิมตรง ๆ

ถ้า attribute เปลี่ยน:

1. ปิด record เดิม เช่น set `valid_to` และ `is_current = false`
2. insert record ใหม่ที่เป็น current version เช่น set `valid_from`, `valid_to = 9999-12-31`, `is_current = true`

ตัวอย่าง:

```text
C001 | Alice | Bangkok    | 2026-01-01 | 2026-05-17 | false
C001 | Alice | Chiang Mai | 2026-05-18 | 9999-12-31 | true
```

> Type 2 = historical dimension table

### Natural key vs surrogate key

ใน dimension table เรามักแยก key ออกเป็น 2 แบบ:

- `customer_id` = natural key / business key จาก source
- `customer_sk` = surrogate key สำหรับแต่ละ version ของ dimension record

ใน lab นี้จะใช้ `sha2(customer_id + valid_from, 256)` เป็น surrogate-like key แบบง่าย ๆ เพื่อให้ผลลัพธ์ deterministic (input เดิม → output เดิมเสมอ) และเหมาะกับการเรียน

ใน production จริง surrogate key อาจมาจาก identity/sequence/key generation strategy ของ platform หรือ warehouse design

Note: `sha2(..., 256)` คือ Spark built-in function สำหรับสร้าง hash value แบบ 256-bit จาก input ที่กำหนด โดย input เดิมจะได้ค่าเดิมเสมอ

### Change detection

สำหรับ SCD2 เราไม่ควรสร้าง history version ใหม่ทุกครั้งที่เจอ update timestamp ใหม่เสมอไป

ควรดูว่า attributes ที่เราสนใจเปลี่ยนจริงไหม เช่น:

- `customer_name`
- `city`
- `loyalty_tier`
- `status`

ใน lab นี้จะสร้าง `attr_hash` จาก attributes เหล่านี้ แล้ว compare กับ current record

> ถ้า hash เท่ากัน = attributes ไม่เปลี่ยน  
> ถ้า hash ไม่เท่ากัน = ต้องสร้าง new version


## Mermaid diagram: SCD Type 1 vs Type 2 Flow

```mermaid
flowchart TB
    A[Source update batch] --> B[Deduplicate latest record per customer_id]

    B --> C1[SCD Type 1 MERGE]
    C1 --> D1[Overwrite matched row]
    C1 --> E1[Insert new customer]
    D1 --> F1[Keep only latest state]
    E1 --> F1

    B --> C2[SCD Type 2 change detection]
    C2 -->|New customer| D2[Insert first current version]
    C2 -->|Changed attributes| E2[Expire old current record]
    E2 --> F2[Insert new current version]
    C2 -->|No attribute change| G2[Skip]
    D2 --> H2[Keep timeline history]
    F2 --> H2
    G2 --> H2
```

Key idea:

> Type 1 ตอบคำถามว่า “current value ตอนนี้คืออะไร” ส่วน Type 2 ตอบคำถามว่า “historical value ณ เวลานั้นคืออะไร”

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 3, Finished, Available, Finished, False)

## Create mock data

เราจะใช้ mock customer dimension 2 ชุด:

1. Initial dimension records = สถานะเริ่มต้นของ customer dimension
2. Source update batch = records ที่มาจาก source รอบใหม่ มีทั้ง changed, unchanged, new customer และ duplicate update ใน batch เดียวกัน

In [2]:
initial_customers_data = [
    ("C001", "Alice", "Bangkok", "silver", "active", "2026-05-01 08:00:00", "crm"),
    ("C002", "Bob", "Bangkok", "silver", "active", "2026-05-02 09:00:00", "crm"),
    ("C003", "Charlie", "Phuket", "gold", "active", "2026-05-03 10:00:00", "crm"),
]

customer_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("loyalty_tier", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

df_initial_customers = (
    spark.createDataFrame(initial_customers_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_initial_customers.show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 4, Finished, Available, Finished, False)

+-----------+-------------+-------+------------+------+-------------------+-------------+
|customer_id|customer_name|city   |loyalty_tier|status|updated_at         |source_system|
+-----------+-------------+-------+------------+------+-------------------+-------------+
|C001       |Alice        |Bangkok|silver      |active|2026-05-01 08:00:00|crm          |
|C002       |Bob          |Bangkok|silver      |active|2026-05-02 09:00:00|crm          |
|C003       |Charlie      |Phuket |gold        |active|2026-05-03 10:00:00|crm          |
+-----------+-------------+-------+------------+------+-------------------+-------------+



In [3]:
source_updates_data = [
    # C001 appears twice in the same source window. We should keep the latest update only.
    ("C001", "Alice", "Chiang Mai", "gold", "active", "2026-05-15 10:00:00", "crm", 1),
    ("C001", "Alice", "Chiang Mai", "platinum", "active", "2026-05-18 08:00:00", "crm", 2),

    # C002 changes status from active to inactive.
    ("C002", "Bob", "Bangkok", "silver", "inactive", "2026-05-16 11:00:00", "crm", 1),

    # C003 has newer updated_at but the tracked attributes are the same.
    ("C003", "Charlie", "Phuket", "gold", "active", "2026-05-16 12:00:00", "crm", 1),

    # C004 is a new customer.
    ("C004", "Diana", "Rayong", "bronze", "active", "2026-05-17 09:00:00", "crm", 1),
]

source_update_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("loyalty_tier", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("source_sequence", T.IntegerType(), True),
])

df_customer_updates_raw = (
    spark.createDataFrame(source_updates_data, source_update_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_customer_updates_raw.orderBy("customer_id", "updated_at").show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 5, Finished, Available, Finished, False)

+-----------+-------------+----------+------------+--------+-------------------+-------------+---------------+
|customer_id|customer_name|city      |loyalty_tier|status  |updated_at         |source_system|source_sequence|
+-----------+-------------+----------+------------+--------+-------------------+-------------+---------------+
|C001       |Alice        |Chiang Mai|gold        |active  |2026-05-15 10:00:00|crm          |1              |
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18 08:00:00|crm          |2              |
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16 11:00:00|crm          |1              |
|C003       |Charlie      |Phuket    |gold        |active  |2026-05-16 12:00:00|crm          |1              |
|C004       |Diana        |Rayong    |bronze      |active  |2026-05-17 09:00:00|crm          |1              |
+-----------+-------------+----------+------------+--------+-------------------+-------------+---------------+



## PySpark code examples

ใน section นี้จะทำ flow หลักของ SCD แบบ step-by-step:

1. Deduplicate source update batch
2. Build Type 1 target table และ run merge
3. Build Type 2 target table
4. Detect changed/new rows
5. Expire old current records และ insert new current versions
6. Validate result ว่า current/history ถูกต้อง

### Example 1: Deduplicate source update batch

ก่อนทำ SCD ควรทำให้ source batch เหลือ latest record ต่อ natural key ก่อน

ในตัวอย่างนี้ใช้:

- `partitionBy("customer_id")`
- `orderBy(updated_at desc, source_sequence desc)`
- `row_number() = 1`

เหตุผลคือ source window อาจมีหลาย update ของ customer เดียวกัน ถ้าไม่ deduplicate ก่อน merge อาจได้ผลลัพธ์ไม่ชัดเจนหรือเกิด duplicate history version

In [4]:
latest_update_window = Window.partitionBy("customer_id").orderBy(
    F.col("updated_at").desc(),
    F.col("source_sequence").desc()
)

df_customer_updates = (
    df_customer_updates_raw
    .withColumn("rn", F.row_number().over(latest_update_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_customer_updates.orderBy("customer_id").show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 6, Finished, Available, Finished, False)

+-----------+-------------+----------+------------+--------+-------------------+-------------+---------------+
|customer_id|customer_name|city      |loyalty_tier|status  |updated_at         |source_system|source_sequence|
+-----------+-------------+----------+------------+--------+-------------------+-------------+---------------+
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18 08:00:00|crm          |2              |
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16 11:00:00|crm          |1              |
|C003       |Charlie      |Phuket    |gold        |active  |2026-05-16 12:00:00|crm          |1              |
|C004       |Diana        |Rayong    |bronze      |active  |2026-05-17 09:00:00|crm          |1              |
+-----------+-------------+----------+------------+--------+-------------------+-------------+---------------+



### Example 2: Create an SCD Type 1 target table

สำหรับ Type 1 table เราเก็บ record ล่าสุดของ customer เท่านั้น

ใน lab นี้จะสร้าง table ชื่อ `day24_dim_customer_type1`

In [5]:
type1_table_name = "day24_dim_customer_type1"

df_type1_initial = df_initial_customers.withColumn("processed_at", F.current_timestamp())

(
    df_type1_initial
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(type1_table_name)
)

spark.read.table(type1_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 7, Finished, Available, Finished, False)

+-----------+-------------+-------+------------+------+-------------------+-------------+--------------------------+
|customer_id|customer_name|city   |loyalty_tier|status|updated_at         |source_system|processed_at              |
+-----------+-------------+-------+------------+------+-------------------+-------------+--------------------------+
|C001       |Alice        |Bangkok|silver      |active|2026-05-01 08:00:00|crm          |2026-06-07 15:16:35.116858|
|C002       |Bob          |Bangkok|silver      |active|2026-05-02 09:00:00|crm          |2026-06-07 15:16:35.116858|
|C003       |Charlie      |Phuket |gold        |active|2026-05-03 10:00:00|crm          |2026-06-07 15:16:35.116858|
+-----------+-------------+-------+------------+------+-------------------+-------------+--------------------------+



### Example 3: Run SCD Type 1 MERGE

Type 1 logic:

- ถ้า `customer_id` match และ source ใหม่กว่า target → update record เดิม
- ถ้าไม่ match → insert customer ใหม่
- ไม่มีการเก็บ old value/history

In [6]:
df_customer_updates.createOrReplaceTempView("vw_day24_customer_updates_type1")

spark.sql(f"""
MERGE INTO {type1_table_name} AS target
USING vw_day24_customer_updates_type1 AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN UPDATE SET
    target.customer_name = source.customer_name,
    target.city = source.city,
    target.loyalty_tier = source.loyalty_tier,
    target.status = source.status,
    target.updated_at = source.updated_at,
    target.source_system = source.source_system,
    target.processed_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    customer_id,
    customer_name,
    city,
    loyalty_tier,
    status,
    updated_at,
    source_system,
    processed_at
) VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.loyalty_tier,
    source.status,
    source.updated_at,
    source.source_system,
    current_timestamp()
)
""")

spark.read.table(type1_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 8, Finished, Available, Finished, False)

+-----------+-------------+----------+------------+--------+-------------------+-------------+--------------------------+
|customer_id|customer_name|city      |loyalty_tier|status  |updated_at         |source_system|processed_at              |
+-----------+-------------+----------+------------+--------+-------------------+-------------+--------------------------+
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18 08:00:00|crm          |2026-06-07 15:16:51.478092|
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16 11:00:00|crm          |2026-06-07 15:16:51.478092|
|C003       |Charlie      |Phuket    |gold        |active  |2026-05-16 12:00:00|crm          |2026-06-07 15:16:51.478092|
|C004       |Diana        |Rayong    |bronze      |active  |2026-05-17 09:00:00|crm          |2026-06-07 15:16:51.478092|
+-----------+-------------+----------+------------+--------+-------------------+-------------+--------------------------+



### Example 4: What Type 1 result means

หลัง Type 1 merge:

- `C001` จะเหลือค่า latest เท่านั้น เช่น `city = Chiang Mai`, `loyalty_tier = platinum`
- `C002` จะถูก update เป็น `inactive`
- `C003` มี update record เข้ามา แต่ business attributes ยังเหมือนเดิม
- `C004` ถูก insert เป็น new customer

Type 1 table เหมาะกับ current-state lookup แต่ไม่เหมาะกับการถาม history ย้อนหลัง

In [7]:
df_type1_current_state = (
    spark.read.table(type1_table_name)
    .select(
        "customer_id",
        "customer_name",
        "city",
        "loyalty_tier",
        "status",
        "updated_at"
    )
    .orderBy("customer_id")
)

df_type1_current_state.show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 9, Finished, Available, Finished, False)

+-----------+-------------+----------+------------+--------+-------------------+
|customer_id|customer_name|city      |loyalty_tier|status  |updated_at         |
+-----------+-------------+----------+------------+--------+-------------------+
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18 08:00:00|
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16 11:00:00|
|C003       |Charlie      |Phuket    |gold        |active  |2026-05-16 12:00:00|
|C004       |Diana        |Rayong    |bronze      |active  |2026-05-17 09:00:00|
+-----------+-------------+----------+------------+--------+-------------------+



### Example 5: Create helper function for attribute hash

สำหรับ SCD2 เราต้องรู้ว่า tracked attributes เปลี่ยนจริงไหม

ใน lab นี้จะใช้ `attr_hash` จาก business attributes ที่เราสนใจ

In [8]:
tracked_attribute_columns = [
    "customer_name",
    "city",
    "loyalty_tier",
    "status",
]

def build_attr_hash(column_names):
    # Replace nulls with a fixed placeholder before hashing.
    return F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(column_name).cast("string"), F.lit("__NULL__")) for column_name in column_names]
        ),
        256
    )

# Tip:
# Use * to unpack the list of column expressions into concat_ws arguments.
# Example: concat_ws("||", *[F.col("city"), F.col("status")])
# is the same as concat_ws("||", F.col("city"), F.col("status")).

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 10, Finished, Available, Finished, False)

### Example 6: Create an SCD Type 2 target table

Type 2 table ต้องมี columns สำหรับ timeline:

- `customer_sk` = surrogate-like key ต่อ version
- `customer_id` = natural key
- `valid_from` = วันที่ version เริ่มมีผล
- `valid_to` = วันที่ version หมดอายุ
- `is_current` = บอกว่า version นี้เป็น current record หรือไม่
- `attr_hash` = ใช้ช่วย compare ว่า tracked attributes เปลี่ยนจริงไหม

ใน lab นี้จะสร้าง table ชื่อ `day24_dim_customer_type2`

In [19]:
type2_table_name = "day24_dim_customer_type2"

far_future_date = "9999-12-31"

# Initial dimension version starts from a fixed date for the lab.
df_type2_initial = (
    df_initial_customers
    .withColumn("valid_from", F.to_date(F.lit("2026-01-01")))
    .withColumn("valid_to", F.to_date(F.lit(far_future_date)))
    .withColumn("is_current", F.lit(True))
    .withColumn("attr_hash", build_attr_hash(tracked_attribute_columns))
    .withColumn("customer_sk", F.sha2(F.concat_ws("||", F.col("customer_id"), F.col("valid_from").cast("string")), 256))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("closed_at", F.lit(None).cast("timestamp"))
    .select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "city",
        "loyalty_tier",
        "status",
        "source_system",
        "updated_at",
        "valid_from",
        "valid_to",
        "is_current",
        "attr_hash",
        "created_at",
        "closed_at"
    )
)

(
    df_type2_initial
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(type2_table_name)
)

spark.read.table(type2_table_name).orderBy("customer_id", "valid_from").show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 21, Finished, Available, Finished, False)

+----------------------------------------------------------------+-----------+-------------+-------+------------+------+-------------+-------------------+----------+----------+----------+----------------------------------------------------------------+--------------------------+---------+
|customer_sk                                                     |customer_id|customer_name|city   |loyalty_tier|status|source_system|updated_at         |valid_from|valid_to  |is_current|attr_hash                                                       |created_at                |closed_at|
+----------------------------------------------------------------+-----------+-------------+-------+------------+------+-------------+-------------------+----------+----------+----------+----------------------------------------------------------------+--------------------------+---------+
|faddfbecf86b75afd73c73d75eeba6becc8608ae0b891aca078cbef482774b8d|C001       |Alice        |Bangkok|silver      |active|crm       

### Example 7: Prepare Type 2 source rows with effective date and hash

สำหรับ source update batch เราจะเพิ่ม:

- `effective_from` จาก `updated_at`
- `attr_hash` จาก tracked attributes

จากนั้น compare กับ current records ใน Type 2 table

In [20]:
df_scd2_source = (
    df_customer_updates
    .withColumn("effective_from", F.to_date("updated_at"))
    .withColumn("attr_hash", build_attr_hash(tracked_attribute_columns))
)

df_scd2_current = (
    spark.read.table(type2_table_name)
    .filter(F.col("is_current") == True)
)

df_scd2_source.select(
    "customer_id",
    "customer_name",
    "city",
    "loyalty_tier",
    "status",
    "effective_from",
    "attr_hash"
).orderBy("customer_id").show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 22, Finished, Available, Finished, False)

+-----------+-------------+----------+------------+--------+--------------+----------------------------------------------------------------+
|customer_id|customer_name|city      |loyalty_tier|status  |effective_from|attr_hash                                                       |
+-----------+-------------+----------+------------+--------+--------------+----------------------------------------------------------------+
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18    |e038d0eef7032feea02ee70a05f870f3ad53ebebbffccaac7cecf93180d12888|
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16    |e679e0688b46ae2bce918f40b98543876f056ec2d8bee485422d2ad5f56c9a5e|
|C003       |Charlie      |Phuket    |gold        |active  |2026-05-16    |2104c4b1ab6e6530dfdcff7f542bae6079315966c1640555e937e6133b81e4cf|
|C004       |Diana        |Rayong    |bronze      |active  |2026-05-17    |7e40efa789545c514d0f28f5091eff17642fa687ca1cde495aa2c874d1617aaf|
+-----------+

### Example 8: Detect new, changed, and unchanged customers

SCD2 ไม่ควร insert new version ทุกครั้งที่มี source update

เราจะแบ่ง source update เป็น 3 กลุ่ม:

- New customer: ไม่มีใน current dimension
- Changed customer: มีใน current dimension แต่ `attr_hash` เปลี่ยน
- Unchanged customer: มีใน current dimension และ `attr_hash` เหมือนเดิม

In [21]:
source_alias = df_scd2_source.alias("source")  # Alias incoming records as source so source columns can be referenced clearly after joins.
current_df = df_scd2_current

source_columns = df_scd2_source.columns

df_new_customers = (
    source_alias
    .join(current_df.select("customer_id"), on="customer_id", how="left_anti")  # Only customer_id is needed from the current table to check whether the source customer is new.
    .select(*[F.col(f"source.{column_name}").alias(column_name) for column_name in source_columns])  # * unpacks list into multiple column-name arguments
)

df_changed_customers = (
    source_alias
    .join(current_df.select("customer_id", "attr_hash").alias("target"), on="customer_id", how="inner")  # Join by customer_id, then compare target.attr_hash.
    .filter(F.col("source.attr_hash") != F.col("target.attr_hash"))
    .select(*[F.col(f"source.{column_name}").alias(column_name) for column_name in source_columns])
)

df_unchanged_customers = (
    source_alias
    .join(current_df.select("customer_id", "attr_hash").alias("target"), on="customer_id", how="inner")
    .filter(F.col("source.attr_hash") == F.col("target.attr_hash"))
    .select(*[F.col(f"source.{column_name}").alias(column_name) for column_name in source_columns])
)

print("New customers:", df_new_customers.count())
print("Changed customers:", df_changed_customers.count())
print("Unchanged customers:", df_unchanged_customers.count())

df_new_customers.select("customer_id", "city", "loyalty_tier", "status", "effective_from").orderBy("customer_id").show(truncate=False)
df_changed_customers.select("customer_id", "city", "loyalty_tier", "status", "effective_from").orderBy("customer_id").show(truncate=False)
df_unchanged_customers.select("customer_id", "city", "loyalty_tier", "status", "effective_from").orderBy("customer_id").show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 23, Finished, Available, Finished, False)

New customers: 1
Changed customers: 2
Unchanged customers: 1
+-----------+------+------------+------+--------------+
|customer_id|city  |loyalty_tier|status|effective_from|
+-----------+------+------------+------+--------------+
|C004       |Rayong|bronze      |active|2026-05-17    |
+-----------+------+------------+------+--------------+

+-----------+----------+------------+--------+--------------+
|customer_id|city      |loyalty_tier|status  |effective_from|
+-----------+----------+------------+--------+--------------+
|C001       |Chiang Mai|platinum    |active  |2026-05-18    |
|C002       |Bangkok   |silver      |inactive|2026-05-16    |
+-----------+----------+------------+--------+--------------+

+-----------+------+------------+------+--------------+
|customer_id|city  |loyalty_tier|status|effective_from|
+-----------+------+------------+------+--------------+
|C003       |Phuket|gold        |active|2026-05-16    |
+-----------+------+------------+------+--------------+



### Example 9: Expire old current records for changed customers

สำหรับ changed customers ต้องปิด record ปัจจุบันเดิมก่อน โดย set:

- `valid_to = effective_from - 1 day`
- `is_current = false`
- `closed_at = current_timestamp()`

หมายเหตุ: lab นี้ใช้ date-level effective period เพื่อให้เห็นภาพง่าย ๆ เช่น version ใหม่เริ่ม `2026-02-15` จึงปิด version เดิมที่ `2026-02-14` เพื่อให้ช่วงเวลาไม่ซ้อนกัน

In [22]:
df_changed_for_expire = (
    df_changed_customers
    .select(
        "customer_id",
        "effective_from"
    )
)

df_changed_for_expire.createOrReplaceTempView("vw_day24_scd2_changed_for_expire")

spark.sql(f"""
MERGE INTO {type2_table_name} AS target
USING vw_day24_scd2_changed_for_expire AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true
WHEN MATCHED THEN UPDATE SET
    target.valid_to = date_sub(source.effective_from, 1),  -- date_sub(date, days): Subtracts days from a date.
    target.is_current = false,
    target.closed_at = current_timestamp()
""")

spark.read.table(type2_table_name).orderBy("customer_id", "valid_from").select(
    "customer_id",
    "city",
    "loyalty_tier",
    "status",
    "valid_from",
    "valid_to",
    "is_current"
).show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 24, Finished, Available, Finished, False)

+-----------+-------+------------+------+----------+----------+----------+
|customer_id|city   |loyalty_tier|status|valid_from|valid_to  |is_current|
+-----------+-------+------------+------+----------+----------+----------+
|C001       |Bangkok|silver      |active|2026-01-01|2026-05-17|false     |
|C002       |Bangkok|silver      |active|2026-01-01|2026-05-15|false     |
|C003       |Phuket |gold        |active|2026-01-01|9999-12-31|true      |
+-----------+-------+------------+------+----------+----------+----------+



### Example 10: Insert new current versions for new and changed customers

หลังจากปิด old current records แล้ว เราต้อง insert current version ใหม่สำหรับ:

- changed customers
- new customers

เพื่อให้การ rerun เฉพาะ cell insert นี้ปลอดภัยขึ้น ใน lab จะใช้ `customer_sk` ทำ left anti join กับ table เดิมก่อน append เพื่อกัน duplicate SCD2 version

Note: แนวคิดสำคัญคือ SCD2 insert ควร rerun แล้วไม่สร้าง version ซ้ำ ใน lab นี้จึงใช้ `customer_sk` ทำ duplicate check ก่อน append แบบง่าย ๆ ส่วน production จริงอาจใช้ `MERGE` หรือ retry / idempotency design ที่รัดกุมกว่านี้


In [23]:
df_scd2_insert_candidates = df_changed_customers.unionByName(df_new_customers)  # unionByName(): Combine DataFrames by matching column names

df_scd2_new_versions = (
    df_scd2_insert_candidates
    .withColumn("valid_from", F.col("effective_from"))
    .withColumn("valid_to", F.to_date(F.lit(far_future_date)))
    .withColumn("is_current", F.lit(True))
    .withColumn("customer_sk", F.sha2(F.concat_ws("||", F.col("customer_id"), F.col("valid_from").cast("string")), 256))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("closed_at", F.lit(None).cast("timestamp"))
    .select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "city",
        "loyalty_tier",
        "status",
        "source_system",
        "updated_at",
        "valid_from",
        "valid_to",
        "is_current",
        "attr_hash",
        "created_at",
        "closed_at"
    )
)

existing_type2_keys = spark.read.table(type2_table_name).select("customer_sk")

df_scd2_rows_to_insert = (
    df_scd2_new_versions
    .join(existing_type2_keys, on="customer_sk", how="left_anti")
)

print("Rows to insert:", df_scd2_rows_to_insert.count())

(
    df_scd2_rows_to_insert
    .write
    .mode("append")
    .format("delta")
    .saveAsTable(type2_table_name)
)

spark.read.table(type2_table_name).orderBy("customer_id", "valid_from").select(
    "customer_id",
    "customer_name",
    "city",
    "loyalty_tier",
    "status",
    "valid_from",
    "valid_to",
    "is_current"
).show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 25, Finished, Available, Finished, False)

Rows to insert: 3
+-----------+-------------+----------+------------+--------+----------+----------+----------+
|customer_id|customer_name|city      |loyalty_tier|status  |valid_from|valid_to  |is_current|
+-----------+-------------+----------+------------+--------+----------+----------+----------+
|C001       |Alice        |Bangkok   |silver      |active  |2026-01-01|2026-05-17|false     |
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18|9999-12-31|true      |
|C002       |Bob          |Bangkok   |silver      |active  |2026-01-01|2026-05-15|false     |
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16|9999-12-31|true      |
|C003       |Charlie      |Phuket    |gold        |active  |2026-01-01|9999-12-31|true      |
|C004       |Diana        |Rayong    |bronze      |active  |2026-05-17|9999-12-31|true      |
+-----------+-------------+----------+------------+--------+----------+----------+----------+



### Example 11: Validate current records and history records

หลังทำ SCD2 ควร validate อย่างน้อย 2 เรื่อง:

1. ต้องมี current record แค่ 1 record ต่อ `customer_id`
2. ต้องเห็น history timeline ของ records ที่เปลี่ยนจริง

In [24]:
df_type2_all = spark.read.table(type2_table_name)

current_record_check = (
    df_type2_all
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("version_count"),
        F.sum(F.col("is_current").cast("int")).alias("current_record_count")
    )
    .orderBy("customer_id")
)

current_record_check.show(truncate=False)

# This should return no rows.
current_record_check.filter(F.col("current_record_count") != 1).show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 26, Finished, Available, Finished, False)

+-----------+-------------+--------------------+
|customer_id|version_count|current_record_count|
+-----------+-------------+--------------------+
|C001       |2            |1                   |
|C002       |2            |1                   |
|C003       |1            |1                   |
|C004       |1            |1                   |
+-----------+-------------+--------------------+

+-----------+-------------+--------------------+
|customer_id|version_count|current_record_count|
+-----------+-------------+--------------------+
+-----------+-------------+--------------------+



### Example 12: Point-in-time query from SCD Type 2

ประโยชน์หลักของ SCD2 คือทำให้เราสามารถ query ย้อนหลังได้ว่า dimension value เป็นอะไร ณ วันที่สนใจ

ตัวอย่างนี้จะ query customer dimension as of `2026-05-16`

In [25]:
as_of_date = "2026-05-16"

df_customer_as_of = (
    spark.read.table(type2_table_name)
    .filter(
        (F.to_date(F.lit(as_of_date)) >= F.col("valid_from"))
        & (F.to_date(F.lit(as_of_date)) <= F.col("valid_to"))  # Keep version where valid_from <= as_of_date <= valid_to.
    )
    .select(
        "customer_id",
        "customer_name",
        "city",
        "loyalty_tier",
        "status",
        "valid_from",
        "valid_to",
        "is_current"
    )
    .orderBy("customer_id")
)

df_customer_as_of.show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 27, Finished, Available, Finished, False)

+-----------+-------------+-------+------------+--------+----------+----------+----------+
|customer_id|customer_name|city   |loyalty_tier|status  |valid_from|valid_to  |is_current|
+-----------+-------------+-------+------------+--------+----------+----------+----------+
|C001       |Alice        |Bangkok|silver      |active  |2026-01-01|2026-05-17|false     |
|C002       |Bob          |Bangkok|silver      |inactive|2026-05-16|9999-12-31|true      |
|C003       |Charlie      |Phuket |gold        |active  |2026-01-01|9999-12-31|true      |
+-----------+-------------+-------+------------+--------+----------+----------+----------+



## Quick recap

| Question | Answer |
|---|---|
| SCD Type 1 เก็บอะไร? | เก็บ current state ล่าสุด โดย overwrite record เดิม |
| SCD Type 2 เก็บอะไร? | เก็บ history หลาย version พร้อม effective period |
| Natural key คืออะไร? | Business key จาก source เช่น `customer_id` |
| Surrogate key ใช้ทำอะไร? | แยกแต่ละ version ของ dimension record ออกจากกัน |
| ทำไมต้องมี `is_current`? | เพื่อหา current record ได้ง่าย และคุมให้มี current แค่ 1 record ต่อ key |
| ทำไมต้อง deduplicate source batch ก่อน SCD? | เพื่อให้แต่ละ business key มี update ล่าสุดเพียง record เดียวก่อน merge/insert |
| SCD2 ควรสร้าง version ใหม่เมื่อไร? | เมื่อ tracked attributes เปลี่ยนจริง ไม่ใช่แค่ `updated_at` ใหม่เสมอไป |

## Exercises

### Exercise 1: Apply another Type 1 update batch

ลองเพิ่ม source update batch อีกชุดแล้ว merge เข้า `day24_dim_customer_type1`

Requirements:

- update `C004` จาก `bronze / active` เป็น `silver / inactive`
- insert customer ใหม่ `C005`
- ใช้ `MERGE INTO` แบบ Type 1
- แสดง current-state table หลัง merge

Expected result:

- `C004` เหลือค่าล่าสุดเท่านั้น
- `C005` ถูกเพิ่มเข้ามาใน Type 1 table
- Type 1 table ไม่มี history rows

In [26]:
exercise_type1_updates_data = [
    ("C004", "Diana", "Rayong", "silver", "inactive", "2026-05-20 10:00:00", "crm", 1),
    ("C005", "Ethan", "Bangkok", "bronze", "active", "2026-05-21 09:00:00", "crm", 1),
]

df_type1_exercise_updates = (
    spark.createDataFrame(exercise_type1_updates_data, source_update_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_type1_exercise_updates.createOrReplaceTempView("vw_day24_type1_exercise_updates")

spark.sql(f"""
MERGE INTO {type1_table_name} AS target
USING vw_day24_type1_exercise_updates AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN UPDATE SET
    target.customer_name = source.customer_name,
    target.city = source.city,
    target.loyalty_tier = source.loyalty_tier,
    target.status = source.status,
    target.updated_at = source.updated_at,
    target.source_system = source.source_system,
    target.processed_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    customer_id,
    customer_name,
    city,
    loyalty_tier,
    status,
    updated_at,
    source_system,
    processed_at
) VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.loyalty_tier,
    source.status,
    source.updated_at,
    source.source_system,
    current_timestamp()
)
""")

spark.read.table(type1_table_name).orderBy("customer_id").select(
    "customer_id",
    "customer_name",
    "city",
    "loyalty_tier",
    "status",
    "updated_at"
).show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 28, Finished, Available, Finished, False)

+-----------+-------------+----------+------------+--------+-------------------+
|customer_id|customer_name|city      |loyalty_tier|status  |updated_at         |
+-----------+-------------+----------+------------+--------+-------------------+
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18 08:00:00|
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16 11:00:00|
|C003       |Charlie      |Phuket    |gold        |active  |2026-05-16 12:00:00|
|C004       |Diana        |Rayong    |silver      |inactive|2026-05-20 10:00:00|
|C005       |Ethan        |Bangkok   |bronze      |active  |2026-05-21 09:00:00|
+-----------+-------------+----------+------------+--------+-------------------+



### Exercise 2: Query SCD2 dimension as of a business date

ใช้ SCD2 table เพื่อตอบคำถามว่า customer profile เป็นอย่างไร ณ วันที่ `2026-05-18`

Requirements:

- filter ด้วย condition `as_of_date between valid_from and valid_to`
- select business columns และ effective period columns
- order by `customer_id`

Expected result:

- เห็น customer version ที่ valid ณ วันที่ `2026-05-18`
- `C001` ควรเห็น version ใหม่ที่เริ่มมีผลตั้งแต่ `2026-05-18`
- `C002` ควรเห็น version inactive ที่เริ่มมีผลตั้งแต่ `2026-05-16`

In [31]:
exercise_as_of_date = "2026-05-18"

exercise_point_in_time_result = (
    spark.read.table(type2_table_name)
    .filter(
        (F.to_date(F.lit(exercise_as_of_date)) >= F.col("valid_from"))
        & (F.to_date(F.lit(exercise_as_of_date)) <= F.col("valid_to"))
    )
    .select(
        "customer_id",
        "customer_name",
        "city",
        "loyalty_tier",
        "status",
        "valid_from",
        "valid_to",
        "is_current"
    )
    .orderBy("customer_id")
)

exercise_point_in_time_result.show(truncate=False)

StatementMeta(, 85f48db2-05af-4fd0-be1e-4c9647fdeb34, 33, Finished, Available, Finished, False)

+-----------+-------------+----------+------------+--------+----------+----------+----------+
|customer_id|customer_name|city      |loyalty_tier|status  |valid_from|valid_to  |is_current|
+-----------+-------------+----------+------------+--------+----------+----------+----------+
|C001       |Alice        |Chiang Mai|platinum    |active  |2026-05-18|9999-12-31|true      |
|C002       |Bob          |Bangkok   |silver      |inactive|2026-05-16|9999-12-31|true      |
|C003       |Charlie      |Phuket    |gold        |active  |2026-01-01|9999-12-31|true      |
|C004       |Diana        |Rayong    |bronze      |active  |2026-05-17|9999-12-31|true      |
+-----------+-------------+----------+------------+--------+----------+----------+----------+



## Common mistakes

1. **ใช้ Type 1 ทั้งที่ business ต้องการ history**

   ถ้ารายงานต้องย้อนกลับไปดูค่าของ dimension ณ วันที่เกิด transaction การ overwrite แบบ Type 1 จะทำให้ historical reporting ผิดได้

2. **ทำ Type 2 แล้วลืมปิด old current record**

   ถ้า insert record ใหม่แต่ไม่ set old record เป็น `is_current = false` จะทำให้ key เดียวมี current record มากกว่า 1 row

3. **ปิด old record แล้วลืม insert new current version**

   ถ้าปิด record เดิมแต่ไม่ insert version ใหม่ customer key นั้นจะไม่มี current record เหลืออยู่

4. **ไม่ deduplicate source window ก่อน SCD**

   ถ้า source batch มีหลาย update ต่อ key แล้ว merge/insert ทันที อาจเกิดผลลัพธ์ไม่แน่นอนหรือสร้าง history version เพิ่มเกินความจำเป็น

5. **ใช้แค่ `updated_at` เพื่อตัดสินว่าเกิด Type 2 change**

   `updated_at` ใหม่ไม่ได้แปลว่า tracked attributes เปลี่ยนเสมอไป ควร compare attributes ที่ต้องการ track จริง ๆ

6. **กำหนด effective period boundary ไม่ชัด**

   ต้องระบุให้ชัดว่า `valid_to` เป็น inclusive หรือ exclusive

   - Inclusive = นับรวมวันที่ `valid_to` เช่น `valid_from <= as_of_date <= valid_to`
   - Exclusive = ไม่นับรวมวันที่ `valid_to` เช่น `valid_from <= as_of_date < valid_to`

   ใน lab นี้ใช้ inclusive date range เพราะเราเก็บช่วงเวลาเป็นระดับวันที่ และปิด old version ด้วย `valid_to = effective_from - 1 day` ทำให้ query ย้อนหลังด้วย `as_of_date <= valid_to` ได้ตรงไปตรงมาและไม่เกิดช่วงเวลาทับซ้อนกันระหว่างแต่ละ version

7. **สับสนระหว่าง natural key กับ surrogate key**

   `customer_id` บอกว่าเป็น customer คนไหน ส่วน `customer_sk` บอกว่าเป็น version ไหนของ customer คนนั้น


## Expected Output / Success Criteria

เมื่อจบ Day 24 ควรทำได้:

- อธิบายได้ว่า **SCD Type 1** คือการ overwrite current state และไม่เก็บ history
- อธิบายได้ว่า **SCD Type 2** คือการเก็บหลาย version ด้วย effective period
- แยกได้ว่าเมื่อไรควรใช้ Type 1 และเมื่อไรควรใช้ Type 2
- สร้าง source update batch และ deduplicate latest record ต่อ `customer_id` ได้
- ใช้ `MERGE INTO` เพื่อทำ Type 1 update/insert บน Lakehouse table ได้
- สร้าง Type 2 table ที่มี `valid_from`, `valid_to`, `is_current`, `customer_sk` และ `attr_hash` ได้
- detect ได้ว่า record ไหนเป็น new, changed, unchanged จาก current dimension
- expire old current records และ insert new current versions สำหรับ Type 2 ได้
- validate ได้ว่า SCD2 table มี current record แค่ 1 row ต่อ natural key
- query SCD2 table แบบ point-in-time ด้วย business date ได้

## Final takeaway

SCD คือ pattern สำคัญในการจัดการ dimension data ที่เปลี่ยนตามเวลา:

> Type 1 ใช้เก็บค่าล่าสุด ส่วน Type 2 ใช้เก็บประวัติที่ต้อง query ย้อนหลังได้

สิ่งที่ต้องจำสำหรับ Data Engineer:

- เลือก Type 1 หรือ Type 2 ตาม business requirement ไม่ใช่ตามความง่ายของ code
- ก่อนทำ SCD ควร deduplicate source batch ให้เหลือ latest record ต่อ key
- SCD2 ต้องมี current record แค่ 1 record ต่อ natural key
- Change detection ควรดู tracked attributes ไม่ใช่ดูแค่ `updated_at`
- Point-in-time query คือเหตุผลสำคัญที่ทำให้ SCD2 มีคุณค่าใน reporting

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day24_dim_customer_type1")
# spark.sql("DROP TABLE IF EXISTS day24_dim_customer_type2")